# prepare_dataset

Ten notebook służy do przygotowania finalnego zbioru do modelowania:
- wybór jednego targetu / białka,
- zbudowanie `model_df`,
- opcjonalne ograniczenie zbioru do ok. 10k rekordów,
- random split,
- scaffold split,
- mały subset do sanity check / overfitu.



## 1. Importy i ustawienia

In [ ]:
!pip install -q chembl-downloader requests pandas pyspark rdkit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.0/37.0 MB 21.0 MB/s eta 0:00:00


In [ ]:
from pathlib import Path
import chembl_downloader
sqlite_path = chembl_downloader.download_extract_sqlite()
from pyspark.sql import SparkSession


spark = (
    SparkSession.builder
    .appName("chembl-eda")
    .master("local[*]")
    .config("spark.jars.packages", "org.xerial:sqlite-jdbc:3.51.1.0")
    .getOrCreate()
)

import pandas as pd

from data_preparation import (
    cast_chembl_tables,
    build_base_table,
    filter_activity_rows,
    summarize_target_candidates,
    build_single_target_regression_dataset,
    quick_modeling_report,
)

from splits import (
    random_split,
    scaffold_split,
    split_summary,
    scaffold_overlap_report,
)

DATA_DIR = Path("prepared_data")
DATA_DIR.mkdir(exist_ok=True)

SEED = 42
STANDARD_TYPE = "IC50"
RELATION = "="
MIN_CONFIDENCE_SCORE = 8.0
ORGANISM = "Homo sapiens"

# Możesz ograniczyć finalny dataset do około 10k rekordów:
MAX_ROWS = 10_000

# Rozmiar malutkiego subsetu do sanity check / overfitu
TINY_SUBSET_SIZE = 128


## 2. Wczytanie surowych tabel

Po tej komórce mają istnieć zmienne:
- `activities`
- `assays`
- `targets`
- `structures`


In [ ]:
# kod ładowania tabel z EDA.
# Przykład:

#
# Po uruchomieniu tej komórki powinny istnieć 4 DataFrame'y:
# activities, assays, targets, structures
# w nowszych wersjach chembl nazwy kolumn moga sie roznic

jdbc_url = f"jdbc:sqlite:{sqlite_path}"
props = {"driver": "org.sqlite.JDBC"}

def t(name, cols=None, custom_schema=None):
    reader = (spark.read.format("jdbc")
              .option("url", jdbc_url)
              .option("dbtable", name)
              .option("driver", "org.sqlite.JDBC"))
    if custom_schema:
        reader = reader.option("customSchema", custom_schema)

    df = reader.load()
    return df.select(*cols) if cols else df


activities = t(
    "activities",
    cols=[
        "activity_id","assay_id","molregno",
        "standard_type","standard_relation","standard_units",
        "standard_value","pchembl_value"
    ],
    custom_schema="""
        activity_id BIGINT,
        assay_id BIGINT,
        molregno BIGINT,
        standard_value DOUBLE,
        pchembl_value DOUBLE
    """
)

assays0 = t("assays")
assays = (assays0
          .withColumnRenamed("tid", "target_id")
          .select("assay_id", "target_id", "assay_type", "confidence_score"))

targets0 = t("target_dictionary")

targets = (targets0
           .withColumnRenamed("tid", "target_id")
           .withColumnRenamed("chembl_id", "target_chembl_id")
           .select("target_id", "target_chembl_id", "pref_name", "target_type", "organism"))


## 3. Cast i tabela bazowa

In [ ]:
activities_c, assays_c, targets_c, structures_c = cast_chembl_tables(
    activities, assays, targets, structures
)

base = build_base_table(activities_c, assays_c, targets_c, structures_c)
base.printSchema()


root
 |-- molregno: long (nullable = true)
 |-- target_id: long (nullable = true)
 |-- assay_id: long (nullable = true)
 |-- activity_id: long (nullable = true)
 |-- standard_type: string (nullable = true)
 |-- standard_relation: string (nullable = true)
 |-- standard_units: string (nullable = true)
 |-- standard_value: double (nullable = true)
 |-- pchembl_value: double (nullable = true)
 |-- assay_type: string (nullable = true)
 |-- confidence_score: double (nullable = true)
 |-- target_chembl_id: string (nullable = true)
 |-- pref_name: string (nullable = true)
 |-- target_type: string (nullable = true)
 |-- organism: string (nullable = true)
 |-- canonical_smiles: string (nullable = true)
 |-- standard_inchi_key: string (nullable = true)



## 4. Wybór kandydatów na target

Najpierw filtrujemy sensowne rekordy, a potem patrzymy, które targety mają najwięcej danych.


In [ ]:
filtered = filter_activity_rows(
    base,
    standard_type=STANDARD_TYPE,
    relation=RELATION,
    min_confidence_score=MIN_CONFIDENCE_SCORE,
    organism=ORGANISM,
)

candidate_targets = summarize_target_candidates(filtered, top_n=30)
candidate_targets.show(30, truncate=False)


+---------+----------------+------------------------------------------------------------------------------+--------------+------------+-----+
|target_id|target_chembl_id|pref_name                                                                     |target_type   |organism    |count|
+---------+----------------+------------------------------------------------------------------------------+--------------+------------+-----+
|100079   |CHEMBL2041      |Proto-oncogene tyrosine-protein kinase receptor Ret                           |SINGLE PROTEIN|Homo sapiens|24164|
|9        |CHEMBL203       |Epidermal growth factor receptor                                              |SINGLE PROTEIN|Homo sapiens|18897|
|100097   |CHEMBL5251      |Tyrosine-protein kinase BTK                                                   |SINGLE PROTEIN|Homo sapiens|16545|
|10938    |CHEMBL2971      |Tyrosine-protein kinase JAK2                                                  |SINGLE PROTEIN|Homo sapiens|14990|
|10980

## 5. Wybów targetu


In [ ]:
TARGET_ID = 100097


## 6. Budowa finalnego datasetu modelowego

In [ ]:
model_df_spark = build_single_target_regression_dataset(
    activities_c,
    assays_c,
    targets_c,
    structures_c,
    target_id=TARGET_ID,
    standard_type=STANDARD_TYPE,
    relation=RELATION,
    min_confidence_score=MIN_CONFIDENCE_SCORE,
    organism=ORGANISM,
)

report = quick_modeling_report(model_df_spark)
report


{'rows': 9400,
 'unique_molecules': 9400,
 'unique_smiles': 9400,
 'target_summary': {'count': '9400',
  'mean': '7.498495743627373',
  'stddev': '1.1753833665166515',
  'min': '3.070581074285707',
  'max': '12.585026652029182'}}

In [ ]:
model_df = model_df_spark.toPandas()
print(model_df.shape)
model_df.head()


(9400, 9)


,molregno,canonical_smiles,standard_inchi_key,target_id,target_chembl_id,pref_name,y,ic50_nM_median,n_measurements
0,37694,CCN(CC)C/C=C/c1nc(O)c2c(ccc3nc(Nc4c(Cl)cccc4Cl...,BPLTVYXMXOYIBK-DHZHZOJOSA-N,100097,CHEMBL5251,Tyrosine-protein kinase BTK,9.397940,0.40,1
1,69049,CN1CCN([C@H]2CC[C@H](n3nc(-c4ccc(Oc5ccccc5)cc4...,RLVCBYBGVBOVFV-HZCBDIJESA-N,100097,CHEMBL5251,Tyrosine-protein kinase BTK,9.036212,0.92,1
2,153087,Oc1n[nH]c(-c2ccccc2)n1,FFSXNTGAFSVILG-UHFFFAOYSA-N,100097,CHEMBL5251,Tyrosine-protein kinase BTK,3.300000,501187.23,1
3,346611,Cc1cc(C)c2c(N)c(C(N)=O)sc2n1,SDMLKCQDZJOSDN-UHFFFAOYSA-N,100097,CHEMBL5251,Tyrosine-protein kinase BTK,5.387216,4100.00,1
4,350235,COc1cc(C)c(Sc2cnc(NC(=O)c3ccc(CNC(C)C(C)(C)C)c...,ZHXNIYGJAOPMSO-UHFFFAOYSA-N,100097,CHEMBL5251,Tyrosine-protein kinase BTK,5.387216,4100.00,3


## 7. Opcjonalne ograniczenie do ~10k rekordów

Jeśli zbiór jest większy, można go na razie przyciąć.
Na start najprościej zrobić losową próbkę z ustalonym ziarnem.


In [ ]:
if MAX_ROWS is not None and len(model_df) > MAX_ROWS:
    model_df = model_df.sample(n=MAX_ROWS, random_state=SEED).reset_index(drop=True)

print("Final dataset shape after optional cap:", model_df.shape)
model_df[["canonical_smiles", "y"]].head()


Final dataset shape after optional cap: (9400, 9)


,canonical_smiles,y
0,CCN(CC)C/C=C/c1nc(O)c2c(ccc3nc(Nc4c(Cl)cccc4Cl...,9.397940
1,CN1CCN([C@H]2CC[C@H](n3nc(-c4ccc(Oc5ccccc5)cc4...,9.036212
2,Oc1n[nH]c(-c2ccccc2)n1,3.300000
3,Cc1cc(C)c2c(N)c(C(N)=O)sc2n1,5.387216
4,COc1cc(C)c(Sc2cnc(NC(=O)c3ccc(CNC(C)C(C)(C)C)c...,5.387216


## 8. Zapis pełnego datasetu

In [ ]:
full_dataset_path = DATA_DIR / "chembl_ic50_model_dataset.csv"
model_df.to_csv(full_dataset_path, index=False)
print(full_dataset_path)


prepared_data/chembl_ic50_model_dataset.csv


## 9. Random split

In [ ]:
random_res = random_split(
    model_df,
    train_size=0.8,
    val_size=0.1,
    test_size=0.1,
    seed=SEED,
)

print(split_summary(random_res))


   split  n_rows  fraction
0  train    7520       0.8
1    val     940       0.1
2   test     940       0.1


## 10. Scaffold split

In [ ]:
scaffold_res = scaffold_split(
    model_df,
    smiles_col="canonical_smiles",
    train_size=0.8,
    val_size=0.1,
    test_size=0.1,
    seed=SEED,
)

print(split_summary(scaffold_res))
print(scaffold_overlap_report(scaffold_res))


   split  n_rows  fraction
0  train    7520       0.8
1    val     940       0.1
2   test     940       0.1
{'train_val_overlap': 0, 'train_test_overlap': 0, 'val_test_overlap': 0}


## 11. Zapis splitów do plików

In [ ]:
random_res.train_df.to_csv(DATA_DIR / "train_random.csv", index=False)
random_res.val_df.to_csv(DATA_DIR / "val_random.csv", index=False)
random_res.test_df.to_csv(DATA_DIR / "test_random.csv", index=False)

scaffold_res.train_df.to_csv(DATA_DIR / "train_scaffold.csv", index=False)
scaffold_res.val_df.to_csv(DATA_DIR / "val_scaffold.csv", index=False)
scaffold_res.test_df.to_csv(DATA_DIR / "test_scaffold.csv", index=False)

print("Saved all split files to:", DATA_DIR.resolve())


Saved all split files to: /content/prepared_data


## 12. Tiny subset do sanity check / overfitu

To będzie bardzo mały train set do pierwszego eksperymentu z MLP.
Na nim sprawdzimy, czy model umie zbić train loss bardzo nisko.


In [ ]:
tiny_train_df = random_res.train_df.sample(
    n=min(TINY_SUBSET_SIZE, len(random_res.train_df)),
    random_state=SEED,
).reset_index(drop=True)

tiny_train_path = DATA_DIR / "train_random_tiny.csv"
tiny_train_df.to_csv(tiny_train_path, index=False)

print("Tiny subset shape:", tiny_train_df.shape)
print("Saved tiny subset to:", tiny_train_path)
tiny_train_df.head()


Tiny subset shape: (128, 9)
Saved tiny subset to: prepared_data/train_random_tiny.csv


,molregno,canonical_smiles,standard_inchi_key,target_id,target_chembl_id,pref_name,y,ic50_nM_median,n_measurements
0,1831174,CC(=O)c1ccc(-c2cc3c(-c4cccc(-n5ccc6cc(C7CC7)cc...,VEVJCCNECRPZOO-UHFFFAOYSA-N,100097,CHEMBL5251,Tyrosine-protein kinase BTK,7.309804,41.7,2
1,2503000,CC(C)(C)c1ccc(C(=O)NCc2ccc(-c3ncnc4[nH]ccc34)c...,ODLDMAFDYHTWSE-UHFFFAOYSA-N,100097,CHEMBL5251,Tyrosine-protein kinase BTK,6.847712,142.0,1
2,3032219,C=CC(=O)N1CCN(C(=O)c2cc(Sc3cnc(NC(=O)c4ccc(N5C...,SFULEOXIPYUXER-UHFFFAOYSA-N,100097,CHEMBL5251,Tyrosine-protein kinase BTK,7.259637,55.0,1
3,2567426,CNC(=O)[C@H]1Cc2c([nH]c3ccccc23)CN1C(=O)C[C@@H...,UMWWEZMNPBLTBD-SHVQYXQLSA-N,100097,CHEMBL5251,Tyrosine-protein kinase BTK,6.634512,232.0,1
4,2003121,CN1CCC(c2ccc(Nc3cc(-c4cc(F)cc(N5CCc6c(sc7c6CC(...,CWCCKSPETMTVKY-UHFFFAOYSA-N,100097,CHEMBL5251,Tyrosine-protein kinase BTK,8.657577,2.2,1


## 13. Szybka kontrola

Tu tylko upewniamy się, że wszystko wygląda sensownie:
- nie ma pustych SMILES,
- target `y` istnieje,
- liczności splitów są sensowne.


In [ ]:
print("Null SMILES in full dataset:", model_df["canonical_smiles"].isna().sum())
print("Null y in full dataset:", model_df["y"].isna().sum())

print("\nRandom split sizes:")
print(len(random_res.train_df), len(random_res.val_df), len(random_res.test_df))

print("\nScaffold split sizes:")
print(len(scaffold_res.train_df), len(scaffold_res.val_df), len(scaffold_res.test_df))


Null SMILES in full dataset: 0
Null y in full dataset: 0

Random split sizes:
7520 940 940

Scaffold split sizes:
7520 940 940


## Co dalej?

Po uruchomieniu tego notebooka będziesz mieć gotowe pliki:
- `prepared_data/chembl_ic50_model_dataset.csv`
- `prepared_data/train_random.csv`
- `prepared_data/val_random.csv`
- `prepared_data/test_random.csv`
- `prepared_data/train_scaffold.csv`
- `prepared_data/val_scaffold.csv`
- `prepared_data/test_scaffold.csv`
- `prepared_data/train_random_tiny.csv`

Następny krok:
- w `mlp_model.py` zdefiniować MLP i funkcję `smiles -> Morgan fingerprint`,
- w `train_mlp.ipynb` zacząć od `train_random_tiny.csv` i sprawdzić, czy model potrafi overfitować.
